In [1]:
import numpy as np

In [3]:
data = np.load("2eq8_ensemble_theseus_backbone.npy")

In [4]:
data.shape

(501, 39, 4, 3)

In [10]:
#3d visualize using plotly
import plotly.express as px
atom = 0 # blue
x = data[0, :, atom, 0]
y = data[0, :, atom, 1]
z = data[0, :, atom, 2]

#atom 1 green:
atom = 1 # green
x1 = data[0, :, atom, 0]
y1 = data[0, :, atom, 1]
z1 = data[0, :, atom, 2]

#atom 2 green:
atom = 2 # green
x2 = data[0, :, atom, 0]
y2 = data[0, :, atom, 1]
z2 = data[0, :, atom, 2]

#atom 3 red:
atom = 3 # red
x3 = data[0, :, atom, 0]
y3 = data[0, :, atom, 1]
z3 = data[0, :, atom, 2]

fig = px.scatter_3d(x=x, y=y, z=z, color_discrete_sequence=["blue"], title="Atom 0")
fig.add_scatter3d(x=x1, y=y1, z=z1, mode="markers", marker=dict(color="green"), name="Atom 1")
fig.add_scatter3d(x=x2, y=y2, z=z2, mode="markers", marker=dict(color="green"), name="Atom 2")
fig.add_scatter3d(x=x3, y=y3, z=z3, mode="markers", marker=dict(color="red"), name="Atom 3")
fig.show()

In [6]:
from Bio.PDB import PDBParser, PDBIO, Select

class Backbone(Select):
    def accept_atom(self, atom):
        return atom.get_name() in {"N", "CA", "C", "O"}

s = PDBParser(QUIET=True).get_structure("x", "2eq8_ensemble_theseus.pdb")
io = PDBIO()
io.set_structure(s)
io.save("template_backbone.pdb", Backbone())

In [ ]:
import numpy as np
from Bio.PDB import PDBParser, PDBIO

coords = np.load("2eq8_ensemble_theseus_backbone.npy")
frame = coords[0].reshape(-1, 3)

structure = PDBParser(QUIET=True).get_structure("x", "template_backbone.pdb")
model = structure[0]

atoms = list(model.get_atoms())[0:len(frame)]  # get only the atoms that correspond to the frame

print("atoms in template:", len(atoms))
print("coords in frame:", len(frame))

assert len(atoms) == len(frame), "Template atom count and coordinate count do not match."

for atom, xyz in zip(atoms, frame):
    atom.set_coord(xyz)

io = PDBIO()
io.set_structure(model)   # save only the modified first model
io.save("out.pdb")

atoms in template: 156
coords in frame: 156


In [45]:
%reload_ext autoreload
%autoreload 2
import numpy2pdb
numpy2pdb.numpy2pdb(coords[0], "template_backbone.pdb", "out.pdb")